# Step 1: Setup and Installation

## Step 1.1: Initialize and Verify the LLM

In this step, we configure the OpenAI language model using `ChatOpenAI`.

First, we load environment variables from the `.env` file to securely access the `OPENAI_API_KEY`.  
We then verify that the API key is available before initializing the model.

Finally, we send a simple test prompt to confirm that the LLM is working correctly.

If the model returns the expected response, it confirms that:
- The API key is properly configured  
- The OpenAI connection is working  
- The LLM is ready for MCP integration

In [ ]:
import os
print("CP 1")
from dotenv import load_dotenv
print("CP 2")
from langchain_openai import ChatOpenAI
print("CP 3")
# Load environment variables from .env file
load_dotenv()
print("CP 4")
# Verify API key is loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables. Please create a .env file with your API key.")
    print("CP 5")
print("CP 6")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("CP 7")
response = llm.invoke("Say 'LLM working'")
print("CP 8")
print(response.content)

## Step 1.2: Logging Setup

In this step, we configure a structured logging system for the lab.

The goal is to:

- Display log messages directly in the notebook output
- Persist the same messages to a file named `lablog.txt`
- Include timestamps and log levels automatically
- Avoid duplicate log entries when the cell is re-run

To ensure clean behavior, we:

- Remove any existing root logger handlers
- Create a dedicated logger for this lab
- Disable log propagation to prevent duplicate messages
- Attach exactly one file handler and one stream handler

This setup guarantees consistent, single-instance logging throughout the notebook.

In [ ]:
import logging

# Clear ROOT logger completely
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# Create custom logger
logger = logging.getLogger("lab_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # <-- CRITICAL (prevents double logging)

# Remove existing handlers if cell is re-run
if logger.hasHandlers():
    logger.handlers.clear()

# Create handlers
file_handler = logging.FileHandler("lablog.txt", mode="a", encoding="utf-8")
stream_handler = logging.StreamHandler()

formatter = logging.Formatter(
    "%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

file_handler.setFormatter(formatter)
stream_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(stream_handler)

def pl(message):
    logger.info(message)

pl("Logging system initialized")

# Step 2: Configure MCP Client

In this step, we configure the MCP client to connect to the LangChain Documentation MCP server.

We specify:

- Server name: `langchain-docs`
- Transport type: `http`
- MCP server URL: `https://docs.langchain.com/mcp`

This step initializes the MCP client.
No tools or resources are loaded yet.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient
import asyncio

# Configure the LangChain documentation MCP server
mcp_client = MultiServerMCPClient({
    "langchain-docs": {
        "transport": "http",
        "url": "https://docs.langchain.com/mcp"
    }
})

pl("MCP client configured for LangChain documentation server")
pl("Server: langchain-docs → https://docs.langchain.com/mcp")

# Step 3: Load MCP Tools into LangChain

In this step, we retrieve the tools exposed by the MCP server and convert them into LangChain-compatible tools.

Using `get_tools()`:

- The MCP client connects to the configured server
- Tools are automatically converted to LangChain tool format
- The client remains stateless (sessions are created internally as needed)

We then verify that the tools are successfully loaded and inspect their names and descriptions.

In [ ]:
# Load tools from the MCP server
# The client is stateless: get_tools() creates ephemeral sessions under the hood
mcp_tools = await mcp_client.get_tools()

print(f"Loaded {len(mcp_tools)} tools from MCP server(s)")
print("\nAvailable tools:")
for tool in mcp_tools:
    pl(f"  - {tool.name}: {tool.description[:80]}...")

In [ ]:
#Logging error example
pl("HTTPStatusError: Server error '500 Internal Server Error' for url 'https://docs.langchain.com/mcp")

-----------------

## Change of Approach: Switching to a Local MCP Server

During Step 3 (loading tools from the LangChain documentation MCP server), the remote endpoint began returning consistent `500 Internal Server Error` responses when handling MCP protocol POST requests.

Although the URL was reachable via a browser (GET request), the MCP protocol requires streamable HTTP POST communication, which was failing server-side. Since this lab will be submitted via GitHub and may be re-run during grading, relying on an unstable external MCP endpoint introduces unnecessary risk.

To ensure:

- Deterministic execution
- Full control over the MCP server
- Reproducibility for grading
- Independence from external infrastructure

I am switching to a minimal local Python-based MCP server.

This approach still fully satisfies the lab requirements:
- MCP server integration
- Tool loading into LangChain
- Agent usage of MCP tools
- Practical demonstration

The only change is infrastructure control — not architectural design.

## Step 3B.1: Define Local MCP Server

In this step, we define a minimal local MCP server using `FastMCP`.

We configure:

- Server name: `Local Lab MCP Server`
- Host: `127.0.0.1` (local machine)
- Port: `8000`

We also register a single MCP tool:

- `add_numbers(a: int, b: int)`  
  A simple function that adds two integers and returns the result.

This step only defines the MCP server and its tool.
The server is not running yet.
No client connection is established at this stage.

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP(
    name="Local Lab MCP Server",
    host="127.0.0.1",
    port=8000
)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b

pl("Local MCP server defined.")

## Step 3B.2: Start Local MCP Server

In this step, we start the previously defined local MCP server.

We:

- Define a `start_server()` function that runs the MCP server
- Use the `streamable-http` transport (required for Jupyter compatibility)
- Start the server inside a background thread using `threading.Thread`
- Set `daemon=True` so it runs without blocking the notebook

The server is now actively running at:

`http://127.0.0.1:8000/mcp`

At this stage, the MCP server is live and ready to accept client connections.
No client has connected yet.

In [ ]:
import threading

def start_server():
    mcp.run(transport="streamable-http")

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

pl("Local MCP server running at http://127.0.0.1:8000")

## Step 3B.3: Create MCP Client

In this step, we configure the MCP client to connect to our local MCP server.

We specify:

- Server name: `local-mcp`
- Transport type: `streamable_http`
- MCP server URL: `http://127.0.0.1:8000/mcp`

We use `MultiServerMCPClient`, which allows managing one or more MCP servers through a unified interface.

This step initializes the MCP client.
No tools or resources are loaded yet.
No connection handshake has been performed at this stage.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "local-mcp": {
        "transport": "streamable_http",
        "url": "http://127.0.0.1:8000/mcp"
    }
})

pl("MCP client created.")

## Step 3B.4: Load Tools from Local MCP Server

In this step, we retrieve the available tools from the connected local MCP server.

We:

- Call `get_tools()` on the MCP client
- Establish the MCP handshake with the server
- Retrieve the list of registered tools

We then print:

- The number of tools loaded
- Each tool’s name and description

This confirms that:

- The MCP server is reachable
- The HTTP transport is working
- The tool registration was successful

At this stage, the tools are available for integration with a LangChain agent.

In [ ]:
mcp_tools = await mcp_client.get_tools()

pl(f"Loaded {len(mcp_tools)} tool(s) from local MCP server")

for tool in mcp_tools:
    print("-", tool.name, ":", tool.description)

# Step 4: Create Agent with MCP Tools

In this step, we create a LangChain agent that can use the tools exposed by the local MCP server.

We:

- Retrieve the available MCP tools using `get_tools()`
- Pass those tools into `create_agent`
- Provide the initialized LLM (`llm`)
- Define a `system_prompt` that guides the agent’s behavior

The MCP tools are now integrated directly into the agent, allowing it to:

- Decide when a tool should be used
- Automatically call the MCP server
- Incorporate the tool’s output into its final response

This step completes the integration between:

- The local MCP server  
- The MCP client  
- The LangChain agent  

The agent is now capable of invoking MCP tools during its reasoning process.

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Get tools from MCP server
tools = await mcp_client.get_tools()

# Create agent with the model and MCP tools
# Use the 'prompt' parameter (not 'state_modifier', which is deprecated)
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant that can answer questions about LangChain, LangGraph, and LangSmith by searching the official documentation. Always provide accurate, up-to-date information based on the documentation."
)

pl(f"Agent created with {len(tools)} MCP tools")
print(f"Tools: {[tool.name for tool in tools]}")

## Step 4.1: Test Agent with MCP Tool Invocation

In this step, we test the MCP-enabled agent with a query that requires tool usage.

We:

- Send a user message: `"What is 7 plus 5?"`
- Invoke the agent asynchronously using `ainvoke()`
- Capture the full result, including intermediate tool calls

During execution, the agent:

- Recognizes that the question requires arithmetic
- Calls the `add_numbers` MCP tool
- Sends the request to the local MCP server
- Receives the computed result
- Returns a final natural language response

This step confirms that:

- The agent can reason about when to use a tool
- The MCP tool is successfully invoked
- The end-to-end MCP ↔ LangChain integration is functioning correctly

In [ ]:
result = await agent.ainvoke({
    "messages": [
        HumanMessage(content="What is 7 plus 5?")
    ]
})

pl(result)

# Step 5: Access MCP Resources

In this step, we check whether the connected MCP server exposes any resources.

We:

- Call `get_resources()` on the MCP client
- Retrieve the list of available read-only resources
- Print the number of resources discovered

Resources in MCP are typically:

- Files
- Documents
- Static data
- Knowledge sources
- Read-only contextual information

Unlike tools, resources are not executed — they are accessed to provide background context to an agent.

---

## Expected Result in This Implementation

Due to time constraints, the local MCP server implemented in this lab does **not** define any resources. It only exposes a single executable tool (`add_numbers`).

Therefore, this step correctly returns:

- `0 resource(s)`

This confirms that the server does not provide any contextual data sources beyond tools.

---

## How Resources Could Have Been Added

If extended, resources could be added to the MCP server by:

- Registering static data sources (e.g., documents, text files, configuration data)
- Exposing structured information through MCP resource decorators
- Connecting the server to a filesystem directory or database and exposing entries as resources

For example, a practical extension could include:

- A document analysis server exposing files as resources
- A knowledge base server exposing policy documents
- A database-backed server exposing queryable datasets as read-only resources

These resources could then be retrieved using `read_resource()` and injected into the agent’s context.

---

This step completes verification of MCP resource availability.
In this minimal implementation, the architecture demonstrates tool integration only.

In [ ]:
resources = await mcp_client.get_resources()

pl(f"Loaded {len(resources)} resource(s)")
for r in resources:
    print("-", r)

# Step 6: Build a Complete MCP-Enabled Agent

In this final step, we validate the complete MCP–LangChain integration by executing a real query through the agent.

We:

- Send a natural language request: `"Calculate 21 plus 34 using your tools."`
- Invoke the agent asynchronously
- Print the final response returned by the agent

During execution, the agent:

- Interprets the request
- Decides that arithmetic is required
- Calls the `add_numbers` MCP tool
- Sends the request to the local MCP server
- Receives the computed result
- Produces a final natural language answer

This confirms that:

- The MCP server is operational
- The MCP client is correctly configured
- The agent can reason about when to use a tool
- The tool invocation flows end-to-end through MCP
- The agent integrates tool output into its final response

This step demonstrates a complete, functional MCP-enabled LangChain agent implementation.

---

## Reflection

Looking back, the implementation could have been slightly more creative by adding additional tools such as subtraction, multiplication, and division.

However, under time pressure and after encountering infrastructure issues, the priority was to deliver a reliable, bare-bones working solution that demonstrated the core MCP–LangChain integration principles.

The objective was correctness, stability, and successful delivery — not feature expansion.

In [ ]:
pl("Testing MCP-enabled agent...\n")

result = await agent.ainvoke({
    "messages": [
        HumanMessage(content="Calculate 21 plus 34 using your tools.")
    ]
})

print("\nFinal Answer:")
print(result["messages"][-1].content)

# Final Conclusion

This lab was implemented using a **bare-bones (stripped-down) approach** to focus strictly on the core objective: integrating an MCP server with a LangChain agent and demonstrating tool usage successfully.

Initially, I encountered an HTTP 500 error when attempting to connect to a hosted LangChain MCP endpoint. This issue originated from the external server side and was outside my local environment’s control. Rather than spending excessive time debugging third-party infrastructure, I made a deliberate architectural decision to implement a minimal local MCP server to regain full control over the system.

---

## Key Technical Challenges

- Correctly defining the MCP server with explicit host and port configuration.
- Ensuring the MCP client connected to the correct `/mcp` path rather than the root endpoint.
- Isolating LLM configuration issues that temporarily diverted debugging efforts.
- Distinguishing between protocol-level issues and external infrastructure instability.

Each of these reinforced the importance of careful configuration and controlled debugging when working with distributed AI systems.

---

## Lessons Learned

1. **Control Your Environment**  
   When external infrastructure becomes unstable, reducing dependencies and moving to a controlled local setup can dramatically increase development efficiency.

2. **Not All Failures Are Local Failures**  
   A 500 error from a hosted service does not necessarily indicate an implementation issue. Proper diagnosis requires distinguishing between local misconfiguration and external system failure.

3. **Single Points of Failure Matter**  
   Any external dependency — whether an MCP server, API provider, or cloud service — can become a single point of failure. Enterprise-grade systems require mitigation strategies such as redundancy, fallback logic, and graceful degradation.

4. **Architecture Over Tools**  
   The core challenge in AI system integration is not prompts or agents — it is dependency management and system boundaries.

---

## Final Implementation

The final solution included:

- A minimal local MCP server  
- A single functional tool  
- HTTP transport integration  
- MCP client connection via langchain-mcp-adapters  
- A LangChain agent successfully invoking the MCP tool  

This demonstrated the essential MCP–LangChain integration flow in a clean, controlled, and functional manner.

Optional features such as MCP resources and multi-server orchestration were intentionally deferred to maintain focus and deliver within time constraints.

I plan to revisit this lab in the future to expand the implementation and explore more advanced MCP capabilities.